In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

In [2]:
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

from src.data_loader import load_raw_data

In [3]:
RAW_DATA_DIR = BASE_DIR / "data" / "raw"

data = load_raw_data(RAW_DATA_DIR)

orders = data["orders"].copy()
order_details = data["order_details"].copy()
products = data["products"].copy()
customers = data["customers"].copy()
employees = data["employees"].copy()

In [4]:
date_columns = [
    "OrderDate",
    "RequiredDate",
    "ShippedDate"
]

for column in date_columns:
    if column in orders.columns:
        orders[column] = pd.to_datetime(
            orders[column],
            errors="coerce"
        )

orders[date_columns].head()

,OrderDate,RequiredDate,ShippedDate
0,2016-07-04,2016-08-01,2016-07-16
1,2016-07-05,2016-08-16,2016-07-10
2,2016-07-08,2016-08-05,2016-07-12
3,2016-07-08,2016-08-05,2016-07-15
4,2016-07-09,2016-08-06,2016-07-11


In [6]:
sales = orders.merge(
    order_details,
    on="OrderID",
    how="inner"
)

sales = sales.merge(
    products,
    on="ProductID",
    how="left",
    suffixes=("", "_Product")
)

sales = sales.merge(
    customers,
    on="CustomerID",
    how="left",
    suffixes=("", "_Customer")
)

sales = sales.merge(
    employees,
    on="EmployeeID",
    how="left",
    suffixes=("", "_Employee")
)

In [7]:
sales.shape

(609283, 54)

In [8]:
sales.head()

,OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,ShipAddress,...,City_Employee,Region_Employee,PostalCode_Employee,Country_Employee,HomePhone,Extension,Photo,Notes,ReportsTo,PhotoPath
0,10248,VINET,5,2016-07-04,2016-08-01,2016-07-16,3,16.75,Vins et alcools Chevalier,59 rue de l-Abbaye,...,London,British Isles,SW1 8JR,UK,(71) 555-4848,3453,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Steven Buchanan graduated from St. Andrews Uni...,2.0,http://accweb/emmployees/buchanan.bmp
1,10248,VINET,5,2016-07-04,2016-08-01,2016-07-16,3,16.75,Vins et alcools Chevalier,59 rue de l-Abbaye,...,London,British Isles,SW1 8JR,UK,(71) 555-4848,3453,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Steven Buchanan graduated from St. Andrews Uni...,2.0,http://accweb/emmployees/buchanan.bmp
2,10248,VINET,5,2016-07-04,2016-08-01,2016-07-16,3,16.75,Vins et alcools Chevalier,59 rue de l-Abbaye,...,London,British Isles,SW1 8JR,UK,(71) 555-4848,3453,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Steven Buchanan graduated from St. Andrews Uni...,2.0,http://accweb/emmployees/buchanan.bmp
3,10249,TOMSP,6,2016-07-05,2016-08-16,2016-07-10,1,22.25,Toms Spezialitäten,Luisenstr. 48,...,London,British Isles,EC2 7JR,UK,(71) 555-7773,428,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Michael is a graduate of Sussex University (MA...,5.0,http://accweb/emmployees/davolio.bmp
4,10249,TOMSP,6,2016-07-05,2016-08-16,2016-07-10,1,22.25,Toms Spezialitäten,Luisenstr. 48,...,London,British Isles,EC2 7JR,UK,(71) 555-7773,428,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Michael is a graduate of Sussex University (MA...,5.0,http://accweb/emmployees/davolio.bmp


In [9]:
sales["Revenue"] = (
    sales["UnitPrice"]
    * sales["Quantity"]
    * (1 - sales["Discount"])
)

sales[
    ["OrderID", "ProductID", "UnitPrice", "Quantity", "Discount", "Revenue"]
].head(10)

,OrderID,ProductID,UnitPrice,Quantity,Discount,Revenue
0,10248,11,14.0,12,0.00,168.00
1,10248,42,9.8,10,0.00,98.00
2,10248,72,34.8,5,0.00,174.00
3,10249,14,18.6,9,0.00,167.40
4,10249,51,42.4,40,0.00,1696.00
5,10250,41,7.7,10,0.00,77.00
6,10250,51,42.4,35,0.15,1261.40
7,10250,65,16.8,15,0.15,214.20
8,10251,22,16.8,6,0.05,95.76
9,10251,57,15.6,15,0.05,222.30


In [10]:
order_totals = (
    sales
    .groupby("OrderID", as_index=False)["Revenue"]
    .sum()
    .rename(columns={"Revenue": "OrderTotal"})
)

sales = sales.merge(
    order_totals,
    on="OrderID",
    how="left"
)

order_item_count = (
    sales
    .groupby("OrderID")["ProductID"]
    .nunique()
    .reset_index(name="OrderItemCount")
)


sales = sales.merge(
    order_item_count,
    on="OrderID",
    how="left"
)

In [11]:
sales.head()

,OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,ShipAddress,...,Country_Employee,HomePhone,Extension,Photo,Notes,ReportsTo,PhotoPath,Revenue,OrderTotal,OrderItemCount
0,10248,VINET,5,2016-07-04,2016-08-01,2016-07-16,3,16.75,Vins et alcools Chevalier,59 rue de l-Abbaye,...,UK,(71) 555-4848,3453,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Steven Buchanan graduated from St. Andrews Uni...,2.0,http://accweb/emmployees/buchanan.bmp,168.0,440.0,3
1,10248,VINET,5,2016-07-04,2016-08-01,2016-07-16,3,16.75,Vins et alcools Chevalier,59 rue de l-Abbaye,...,UK,(71) 555-4848,3453,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Steven Buchanan graduated from St. Andrews Uni...,2.0,http://accweb/emmployees/buchanan.bmp,98.0,440.0,3
2,10248,VINET,5,2016-07-04,2016-08-01,2016-07-16,3,16.75,Vins et alcools Chevalier,59 rue de l-Abbaye,...,UK,(71) 555-4848,3453,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Steven Buchanan graduated from St. Andrews Uni...,2.0,http://accweb/emmployees/buchanan.bmp,174.0,440.0,3
3,10249,TOMSP,6,2016-07-05,2016-08-16,2016-07-10,1,22.25,Toms Spezialitäten,Luisenstr. 48,...,UK,(71) 555-7773,428,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Michael is a graduate of Sussex University (MA...,5.0,http://accweb/emmployees/davolio.bmp,167.4,1863.4,2
4,10249,TOMSP,6,2016-07-05,2016-08-16,2016-07-10,1,22.25,Toms Spezialitäten,Luisenstr. 48,...,UK,(71) 555-7773,428,/9j/4AAQSkZJRgABAgAAZABkAAD/7AARRHVja3kAAQAEAA...,Michael is a graduate of Sussex University (MA...,5.0,http://accweb/emmployees/davolio.bmp,1696.0,1863.4,2


In [21]:
order_quantity = (
    sales
    .groupby("OrderID")["Quantity"]
    .sum()
    .reset_index(name="OrderQuantity")
)

sales = sales.merge(
    order_quantity,
    on="OrderID",
    how="left"
)

In [13]:
sales["DaysToShip"] = (
    sales["ShippedDate"] - sales["OrderDate"]
).dt.days

sales[
    ["OrderID", "OrderDate", "ShippedDate", "DaysToShip"]
].head()

,OrderID,OrderDate,ShippedDate,DaysToShip
0,10248,2016-07-04,2016-07-16,12.0
1,10248,2016-07-04,2016-07-16,12.0
2,10248,2016-07-04,2016-07-16,12.0
3,10249,2016-07-05,2016-07-10,5.0
4,10249,2016-07-05,2016-07-10,5.0


In [14]:
sales["ShipDelayDays"] = (
    sales["ShippedDate"] - sales["RequiredDate"]
).dt.days

sales["IsLate"] = sales["ShipDelayDays"] > 0


In [15]:
sales["OrderValueCategory"] = pd.qcut(
    sales["OrderTotal"],
    q=3,
    labels=["Small", "Medium", "Large"],
    duplicates="drop"
)

sales[
    ["OrderID", "OrderTotal", "OrderValueCategory"]
].drop_duplicates("OrderID").head(20)

,OrderID,OrderTotal,OrderValueCategory
0,10248,440.000,Small
3,10249,1863.400,Small
5,10250,1552.600,Small
8,10251,654.060,Small
11,10252,3597.900,Small
14,10253,1444.800,Small
17,10254,556.620,Small
20,10255,2490.500,Small
24,10256,517.800,Small
26,10257,1119.900,Small


In [16]:
customer_order_count = (
    sales
    .groupby("CustomerID")["OrderID"]
    .nunique()
    .reset_index(name="CustomerOrderCount")
)

sales = sales.merge(
    customer_order_count,
    on="CustomerID",
    how="left"
)

In [17]:
customer_revenue = (
    sales
    .groupby("CustomerID")["Revenue"]
    .sum()
    .reset_index(name="CustomerLifetimeRevenue")
)

sales = sales.merge(
    customer_revenue,
    on="CustomerID",
    how="left"
)

In [22]:
sales[
    [
        "OrderID",
        "CustomerID",
        "ProductID",
        "Revenue",
        "OrderTotal",
        "OrderItemCount",
        "OrderQuantity",
        "DaysToShip",
        "IsLate",
        "CustomerOrderCount",
        "CustomerLifetimeRevenue"
    ]
].head(20)

,OrderID,CustomerID,ProductID,Revenue,OrderTotal,OrderItemCount,OrderQuantity,DaysToShip,IsLate,CustomerOrderCount,CustomerLifetimeRevenue
0,10248,VINET,11,168.00,440.00,3,27,12.0,False,158,4332214.28
1,10248,VINET,42,98.00,440.00,3,27,12.0,False,158,4332214.28
2,10248,VINET,72,174.00,440.00,3,27,12.0,False,158,4332214.28
3,10249,TOMSP,14,167.40,1863.40,2,49,5.0,False,174,4745078.10
4,10249,TOMSP,51,1696.00,1863.40,2,49,5.0,False,174,4745078.10
5,10250,HANAR,41,77.00,1552.60,3,60,4.0,False,180,5097941.56
6,10250,HANAR,51,1261.40,1552.60,3,60,4.0,False,180,5097941.56
7,10250,HANAR,65,214.20,1552.60,3,60,4.0,False,180,5097941.56
8,10251,VICTE,22,95.76,654.06,3,41,7.0,False,188,5276686.06
9,10251,VICTE,57,222.30,654.06,3,41,7.0,False,188,5276686.06


In [31]:
quality_report = pd.DataFrame({
    "metric": [
        "total_rows",
        "missing_customer",
        "missing_product",
        "missing_employee",
        "non_positive_price",
        "non_positive_quantity",
        "invalid_discount",
        "invalid_shipping_dates"
    ],
    "count": [
        len(sales),
        sales["CustomerID"].isna().sum(),
        sales["ProductName"].isna().sum(),
        sales["EmployeeID"].isna().sum(),
        (sales["UnitPrice"] <= 0).sum(),
        (sales["Quantity"] <= 0).sum(),
        (
            (sales["Discount"] < 0) |
            (sales["Discount"] > 1)
        ).sum(),
        (
            sales["ShippedDate"] < sales["OrderDate"]
        ).sum()
    ]
})

quality_report

,metric,count
0,total_rows,609283
1,missing_customer,0
2,missing_product,0
3,missing_employee,0
4,non_positive_price,0
5,non_positive_quantity,0
6,invalid_discount,0
7,invalid_shipping_dates,0


In [33]:
invalid_rows = (
    (sales["UnitPrice"] <= 0) |
    (sales["Quantity"] <= 0) |
    (sales["Discount"] < 0) |
    (sales["Discount"] > 1) |
    (sales["ShippedDate"] < sales["OrderDate"])
)

In [34]:
invalid_rows.sum()

np.int64(0)

In [35]:
sales_clean = sales.loc[~invalid_rows].copy()

In [36]:
dim_customer = customers[
    [
        "CustomerID",
        "CompanyName",
        "ContactName",
        "ContactTitle",
        "City",
        "Region",
        "Country"
    ]
].copy()


dim_product = products[
    [
        "ProductID",
        "ProductName",
        "SupplierID",
        "CategoryID",
        "UnitPrice",
        "UnitsInStock",
        "UnitsOnOrder",
        "ReorderLevel",
        "Discontinued"
    ]
].copy()

dim_employee = employees[
    [
        "EmployeeID",
        "LastName",
        "FirstName",
        "Title",
        "City",
        "Country"
    ]
].copy()


fact_sales = sales_clean[
    [
        "OrderID",
        "ProductID",
        "CustomerID",
        "EmployeeID",
        "OrderDate",
        "UnitPrice",
        "Quantity",
        "Discount",
        "Revenue",
        "OrderTotal",
        "OrderItemCount",
        "OrderQuantity",
        "DaysToShip",
        "ShipDelayDays",
        "IsLate"
    ]
].copy()

In [37]:
import sqlite3

DB_DIR = BASE_DIR / "data" / "processed"
DB_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "northwind_analytics.db"

conn = sqlite3.connect(DB_PATH)

dim_customer.to_sql(
    "dim_customer",
    conn,
    if_exists="replace",
    index=False
)

dim_product.to_sql(
    "dim_product",
    conn,
    if_exists="replace",
    index=False
)

dim_employee.to_sql(
    "dim_employee",
    conn,
    if_exists="replace",
    index=False
)

fact_sales.to_sql(
    "fact_sales",
    conn,
    if_exists="replace",
    index=False
)

609283

In [38]:
tables_in_db = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    conn
)

tables_in_db

,name
0,dim_customer
1,dim_product
2,dim_employee
3,fact_sales


In [39]:
query = """
SELECT
    SUM(Revenue) AS TotalRevenue
FROM fact_sales;
"""

pd.read_sql_query(query, conn)

,TotalRevenue
0,4.483866e+08


In [40]:
query = """
SELECT
    p.ProductName,
    SUM(f.Revenue) AS TotalRevenue
FROM fact_sales f
JOIN dim_product p
    ON f.ProductID = p.ProductID
GROUP BY
    p.ProductID,
    p.ProductName
ORDER BY
    TotalRevenue DESC;
"""

product_revenue = pd.read_sql_query(
    query,
    conn
)

product_revenue.head(10)

,ProductName,TotalRevenue
0,Côte de Blaye,5.326590e+07
1,Thüringer Rostbratwurst,2.462347e+07
2,Mishi Kobe Niku,1.942304e+07
3,Sir Rodney's Marmalade,1.665381e+07
4,Carnarvon Tigers,1.260467e+07
5,Raclette Courdavault,1.121641e+07
6,Manjimup Dried Apples,1.066477e+07
7,Tarte au sucre,9.952936e+06
8,Ipoh Coffee,9.333375e+06
9,Rössle Sauerkraut,9.252765e+06


In [41]:
query = """
SELECT
    c.CustomerID,
    c.CompanyName,
    SUM(f.Revenue) AS TotalRevenue
FROM fact_sales f
JOIN dim_customer c
    ON f.CustomerID = c.CustomerID
GROUP BY
    c.CustomerID,
    c.CompanyName
ORDER BY
    TotalRevenue DESC;
"""

customer_revenue = pd.read_sql_query(
    query,
    conn
)

customer_revenue.head(10)

,CustomerID,CompanyName,TotalRevenue
0,BSBEV,B's Beverages,6.154115e+06
1,HUNGC,Hungry Coyote Import Store,5.698024e+06
2,RANCH,Rancho grande,5.559110e+06
3,GOURL,Gourmet Lanchonetes,5.552310e+06
4,ANATR,Ana Trujillo Emparedados y helados,5.534357e+06
5,RICAR,Ricardo Adocicados,5.524517e+06
6,FOLIG,Folies gourmandes,5.505503e+06
7,LETSS,Let's Stop N Shop,5.462198e+06
8,LILAS,LILA-Supermercado,5.437438e+06
9,PRINI,Princesa Isabel Vinhos,5.436771e+06


In [42]:
query = """
SELECT
    e.EmployeeID,
    e.FirstName,
    e.LastName,
    SUM(f.Revenue) AS TotalRevenue
FROM fact_sales f
JOIN dim_employee e
    ON f.EmployeeID = e.EmployeeID
GROUP BY
    e.EmployeeID,
    e.FirstName,
    e.LastName
ORDER BY
    TotalRevenue DESC;
"""

employee_revenue = pd.read_sql_query(
    query,
    conn
)

employee_revenue

,EmployeeID,FirstName,LastName,TotalRevenue
0,4,Margaret,Peacock,5.148840e+07
1,5,Steven,Buchanan,5.138646e+07
2,3,Janet,Leverling,5.044557e+07
3,1,Nancy,Davolio,4.965942e+07
4,7,Robert,King,4.965190e+07
5,8,Laura,Callahan,4.928114e+07
6,6,Michael,Suyama,4.913997e+07
7,9,Anne,Dodsworth,4.901968e+07
8,2,Andrew,Fuller,4.831410e+07


In [43]:
conn.close()

In [ ]:
PROCESSED_DIR = BASE_DIR / "data" / "processed"

sales_clean.to_csv(
    PROCESSED_DIR / "sales_clean.csv",
    index=False
)